In [1]:
import numpy as np
from xgboost import XGBClassifier
import lightgbm as lgb
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier
from catboost import CatBoostClassifier
from sklearn.model_selection import train_test_split, ParameterSampler
from sklearn.metrics import *
from sklearn.preprocessing import RobustScaler, StandardScaler, MinMaxScaler, QuantileTransformer
from random import choice
import random

In [ ]:
def generate_random_catboost_params():
    grow_policy = choice(['SymmetricTree', 'Depthwise', 'Lossguide'])
    
    if grow_policy == 'SymmetricTree':
        boosting_type = choice(['Ordered', 'Plain'])
    else:
        boosting_type = 'Plain'  
    
    params = {
        'iterations': choice([100, 200, 300, 500, 1000]),
        'depth': choice([3, 4, 5, 6, 7, 8]),
        'min_data_in_leaf': choice([1, 3, 5, 10, 20]),
        
        'learning_rate': choice([0.01, 0.03, 0.05, 0.08, 0.1, 0.15]),
        
        'l2_leaf_reg': choice([1, 3, 5, 10, 50]),
        'model_size_reg': choice([0, 0.5, 1]),
        
        'subsample': choice([0.7, 0.8, 0.9, 1.0]),
        'random_strength': choice([0, 0.1, 1, 10]),
        'rsm': choice([0.7, 0.8, 0.9, 1.0]),
        
        'one_hot_max_size': choice([2, 10, 20]),
        'has_time': choice([True, False]),
        
        'scale_pos_weight': choice([1, 2, 3, 5, 10]),
        
        'grow_policy': grow_policy,
        'boosting_type': boosting_type,
        
        'thread_count': -1,
        'random_seed': 42,
        'verbose': False,
        'allow_writing_files': False
    }
    
    # Only add max_leaves if using Lossguide
    if grow_policy == 'Lossguide':
        params['max_leaves'] = choice([31, 64, 128])
    
    return params

def generate_random_params():
    return {
        'n_estimators': choice([100, 200, 300, 500]),
        'max_depth': choice([3, 4, 5, 6]),
        'min_child_weight': choice([1, 3, 5]),
        'gamma': choice([0, 0.1, 0.2]),
        
        'learning_rate': choice([0.01, 0.05, 0.1, 0.15]),
        
        'subsample': choice([0.7, 0.8, 0.9]),
        'colsample_bytree': choice([0.7, 0.8, 0.9]),
        
        'reg_alpha': choice([0, 1, 10]),
        'reg_lambda': choice([1, 10, 100]),
        
        'scale_pos_weight': choice([1, 3, 5, 10]),
        
        'tree_method': choice(['hist', 'auto']),

        'verbosity': 0,
        'n_jobs': -1,
        'random_state': 42
    }

def generate_random_params_lightgbm():
    return {
        'n_estimators': choice([100, 200, 300, 500, 1000]),
        'learning_rate': choice([0.01, 0.05, 0.1, 0.15, 0.2]),
        'max_depth': choice([-1, 3, 4, 5, 6, 7, 8]), 
        'num_leaves': choice([31, 63, 127, 255, 511]),
        
        'min_child_samples': choice([5, 10, 20, 30, 50]),
        'min_child_weight': choice([1e-3, 1e-2, 0.1, 1, 5]),
        'min_split_gain': choice([0, 0.1, 0.2, 0.5]),
        
        'subsample': choice([0.6, 0.7, 0.8, 0.9, 1.0]),
        'subsample_freq': choice([0, 1, 5, 10]),
        'colsample_bytree': choice([0.6, 0.7, 0.8, 0.9, 1.0]),
        'reg_alpha': choice([0, 0.1, 1, 5, 10]),
        'reg_lambda': choice([0, 0.1, 1, 5, 10, 50]),
        
        'boosting_type': choice(['gbdt', 'dart']),
        
        'scale_pos_weight': choice([1, 2, 3, 5, 10]),
        
        'subsample_for_bin': choice([20000, 50000, 100000, 200000]),


        'verbosity': -1,
        'n_jobs': -1,
        'random_state': 42


    }

def generate_random_logistic_params():
    return {
        'C': choice([0.001, 0.01, 0.1, 1.0, 10.0, 100.0]),
        'penalty': 'l2',
        'solver': choice(['liblinear', 'lbfgs', 'newton-cg', 'sag', 'saga']),
        'max_iter': choice([100, 200, 500, 1000]),
        'class_weight': choice([None, 'balanced']),
        'random_state': 42
    }

def generate_random_params_decisiontreeclassifier():
    return {
        # Tree Structure and Depth
        'max_depth': choice([5, 10, 15, 20, None]),
        'min_samples_split': choice([2, 5, 10, 20]),
        'min_samples_leaf': choice([1, 5, 10]),
        
        # Split Strategy - USE CLASSIFIER CRITERIA
        'criterion': choice(['gini', 'entropy', 'log_loss']),  # ← CLASSIFIER CRITERIA
        'splitter': choice(['best', 'random']),
        
        # Feature Sampling
        'max_features': choice([None, 'sqrt', 'log2', 0.8, 0.9]),
        
        # Core Configuration
        'random_state': 42
    }

In [ ]:
def create_train_valid_df(df):
    
    div_size = len(df) // 2
    print(div_size)
    
    train_data = df.iloc[:div_size]
    validation_df = df.iloc[div_size:]
    
    train_size = len(train_data) // 3
    train_size *= 2
    print(train_size)
    
    
    df_train = train_data.iloc[:train_size]
    df_test = train_data.iloc[train_size:]
    print(f'train lenght is:{len(df_train)}')
    print(f'test lenght is:{len(df_test)}')

    return df_train, df_test, validation_df

In [ ]:
def optimize_params(df_train, df_test, model_features,
                   model, param_func,
                   min_prec, min_rec, 
                   n_iter):
    
    best_models = []

    X_train = df_train[model_features]
    y_train = df_train['Trades']
    X_test = df_test[model_features]
    y_test = df_test['Trades']

    for i in range(n_iter):
        params = param_func()
        current_model = model(**params)
        current_model.fit(X_train, y_train)
        y_pred = current_model.predict(X_test)
        
        accuracy_1 = accuracy_score(y_test, y_pred)
        precision_1 = precision_score(y_test, y_pred, pos_label = 1)
        recall_1 = recall_score(y_test, y_pred, pos_label = 1)
    
        if precision_1 >= min_prec and recall_1 >= min_rec:
            params = current_model.get_params()
            best_models.append({
                    'model' : model,
                    'params' : params,
                    'precision' : precision_1,
                    'recall' : recall_1
                })
            print(f'model {i} meets criteria: precision = {precision_1}, recall = {recall_1}')
    best_models_df = pd.DataFrame(best_models)
    return best_models_df


def optimize_features(model_features, n_iter, df_train, df_test, model, min_prec, min_rec):

    
    best_features = []
    for i in range(n_iter):

        rand_features = random.sample(model_features, k = random.randint(1, len(model_features)))
    
        X_train_rand = df_train[rand_features]
        y_train_rand = df_train['Trades']
        
        X_test_rand = df_test[rand_features]
        y_test_rand = df_test['Trades']
        
        model.fit(X_train_rand, y_train_rand)
        y_pred_rand = model.predict(X_test_rand)
        
    
        
        rand_precision = precision_score(y_test_rand, y_pred_rand, pos_label = 1)
        rand_recall = recall_score(y_test_rand, y_pred_rand, pos_label = 1)
    
        if rand_precision >= min_prec and rand_recall >= min_rec:
            best_features.append({
                'model': model,
                'n_features': len(rand_features),
                'features': rand_features,
                'precision': rand_precision,
                'recall': rand_recall 
            })
            print(f'new optimization found, iter n: {i}, precision: {rand_precision}, reacll: {rand_recall}, n_features: {len(rand_features)}')

    return best_features


def train_model(df_train, df_test, model, features):
    
    X_train = df_train[features]
    y_train = df_train['Trades']
    X_test = df_test[features]
    y_test = df_test['Trades']
    
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    base_precision = precision_score(y_test, y_pred, pos_label = 1)
    base_recall = recall_score(y_test, y_pred, pos_label = 1)
    
    print(f'model precision is {base_precision}')
    print(f'model recall is {base_recall}')

    return model

def test_model(df, model, features):
    
    X_test = df[features]
    y_test = df['Trades']
    
    y_pred = model.predict(X_test)
    base_precision = precision_score(y_test, y_pred, pos_label = 1)
    base_recall = recall_score(y_test, y_pred, pos_label = 1)
    
    print(f'model precision is {base_precision}')
    print(f'model recall is {base_recall}')

    return base_precision, base_recall

def optimize_model(model_features, param_func, n_iter, df_train, df_test, model, min_prec, min_rec):
    best_models = []
    best_score = 0
    for i in range(n_iter):
        
        rand_params = param_func()
        rand_features = random.sample(model_features, k = random.randint(1, len(model_features)))
    
        X_train_rand = df_train[rand_features]
        y_train_rand = df_train['Trades']
        
        X_test_rand = df_test[rand_features]
        y_test_rand = df_test['Trades']

        current_model = model(**rand_params) 
        
        current_model.fit(X_train_rand, y_train_rand)
        y_pred_rand = current_model.predict(X_test_rand)
    
        
        rand_precision = precision_score(y_test_rand, y_pred_rand, pos_label = 1)
        rand_recall = recall_score(y_test_rand, y_pred_rand, pos_label = 1)
        score = simple_scorer(y_test_rand, y_pred_rand, min_prec = min_prec, min_rec = min_rec)


        if rand_precision >= min_prec and rand_recall >= min_rec and score > best_score:
            best_models.append({
                'model': current_model,
                'n_features': len(rand_features),
                'features': rand_features,
                'params': rand_params,
                'precision': rand_precision,
                'recall': rand_recall,
                'score': score
            })
            best_score = score
            print(f'new optimization found, iter n: {i}, precision: {rand_precision}, reacll: {rand_recall}, n_features: {len(rand_features)}, score: {best_score}')
    best_models_df = pd.DataFrame(best_models)

    return best_models_df

In [ ]:
def train_model_with_scaler(df_train, df_test, model, scaler_type, features):
    
    X_train = df_train[features]
    y_train = df_train['Trades']
    X_test = df_test[features]
    y_test = df_test['Trades']

    scaler = scaler_type()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    base_precision = precision_score(y_test, y_pred, pos_label = 1)
    base_recall = recall_score(y_test, y_pred, pos_label = 1)
    
    print(f'model precision is {base_precision}')
    print(f'model recall is {base_recall}')

    return model

def test_model_with_scaler(df, model, scaler_type, features):
    
    X_test = df[features]
    y_test = df['Trades']

    X_test_scaled = scaler.fit_transform(X_test)
    
    y_pred = model.predict(X_test_scaled)
    base_precision = precision_score(y_test, y_pred, pos_label = 1)
    base_recall = recall_score(y_test, y_pred, pos_label = 1)
    
    print(f'model precision is {base_precision}')
    print(f'model recall is {base_recall}')

    return base_precision, base_recall

def optimize_model_with_scaler(model_features, param_func, n_iter, df_train, df_test, model, scaler_type, min_prec, min_rec):
    
    best_models = []
    best_score = 0
    for i in range(n_iter):
        if i == 1:
            print('iter started correctly, second iter')
        if i % 50 == 0:
            print(f'iter n: {i}')
            
        rand_params = param_func()
        rand_features = random.sample(model_features, k=random.randint(1, len(model_features)))
    
        X_train_rand = df_train[rand_features]
        y_train_rand = df_train['Trades']
        
        X_test_rand = df_test[rand_features]
        y_test_rand = df_test['Trades']
        
        # Create new scaler instance for each iteration
        scaler = scaler_type()
        X_train_scaled = scaler.fit_transform(X_train_rand)
        X_test_scaled = scaler.transform(X_test_rand)

        current_model = model(**rand_params) 
        
        current_model.fit(X_train_scaled, y_train_rand)
        y_pred_rand = current_model.predict(X_test_scaled)
    
        rand_precision = precision_score(y_test_rand, y_pred_rand, pos_label=1)
        rand_recall = recall_score(y_test_rand, y_pred_rand, pos_label=1)
        score = simple_scorer(y_test_rand, y_pred_rand, min_prec = min_prec, min_rec = min_rec)
            
        if rand_precision >= min_prec and rand_recall >= min_rec and score > best_score:
            best_models.append({
                'model': current_model,
                'scaler': scaler,
                'n_features': len(rand_features),
                'features': rand_features,
                'params': rand_params,
                'precision': rand_precision,
                'recall': rand_recall,
                'score': score
            })
            best_score = score
            print(f'New optimization found, iter n: {i}, precision: {rand_precision}, recall: {rand_recall}, n_features: {len(rand_features)}, score: {score}')
    
    best_models_df = pd.DataFrame(best_models)
    
    return best_models_df

def section_validation_with_scaler(df, model, scaler, features):
    step = 100
    performances = []
    tot_samples = len(df)
    
    for start in range(0, tot_samples, step):
        end = min(start + step, tot_samples)  
        
        section_df = df.iloc[start:end]
        
        X_section = section_df[features]
        y_section = section_df['Trades']

        X_section_scaled = scaler.transform(X_section)
    
        #step 1 predict with xgb
        y_pred = model.predict(X_section_scaled)
        precision = precision_score(y_section, y_pred, pos_label = 1)
        y_proba = model.predict_proba(X_section_scaled)[:, 1]
 
        # Calculate metrics for this section only
        precision_section = precision_score(y_section, y_pred, pos_label=1)
        recall_section = recall_score(y_section, y_pred, pos_label=1)
        f1_values = f1_score(y_section, y_pred, average = 'macro')
        winners_num = (section_df['Trades'] == 1).sum()
        losers_num = (section_df['Trades'] == 0).sum()
        performances.append({
            'section': f"{start}-{end}",
            'precision': precision_section,
            'recall': recall_section,
            'f1_values': f1_values,
            'trades_in_section': len(section_df),
            'predicted wins number': y_pred.sum(),
            'winners': winners_num,
            'losers': losers_num
        })

    performances_df = pd.DataFrame(performances)
        
    return performances_df



In [ ]:
def train_meta(df_train, df_test, model_a, model_b, model_c, model_meta, features_a, features_b, features_c):
    
    y_train = df_train['Trades']  
    y_test = df_test['Trades']   

    
    X_train_a = df_train[features_a]
    X_test_a = df_test[features_a]
    
    X_train_b = df_train[features_b]
    X_test_b = df_test[features_b]

    X_train_c = df_train[features_c]
    X_test_c = df_test[features_c]
    
    proba_train_a = model_a.predict_proba(X_train_a)[:, 1]
    proba_train_b = model_b.predict_proba(X_train_b)[:, 1]
    proba_train_c = model_c.predict_proba(X_train_c)[:, 1]

    proba_a = model_a.predict_proba(X_test_a)[:, 1]
    proba_b = model_b.predict_proba(X_test_b)[:, 1]
    proba_c = model_c.predict_proba(X_test_c)[:, 1]
    
    X_meta_train =  np.column_stack([proba_train_a, proba_train_b, proba_train_c])
    X_meta_test = np.column_stack([proba_a, proba_b, proba_c])


    model_meta.fit(X_meta_train, y_train)

    y_pred_meta = model_meta.predict(X_meta_test)

    precision = precision_score(y_test, y_pred_meta, pos_label = 1)
    
    recall = recall_score(y_test, y_pred_meta, pos_label = 1)
    
    print(f'final model precision is: {precision} with a recall of: {recall}')

    return model_meta
    
       

def meta_predictor(model_meta, model_a, model_b, model_c, X_test_a, X_test_b):
    
    X_test_a_reshaped = X_test_a.values.reshape(1,-1)
    X_test_b_reshaped = X_test_b.values.reshape(1,-1)
    X_test_c_reshaped = X_test_c.values.reshape(1,-1)

    proba_a = model_a.predict_proba(X_test_a_reshaped)[:, 1]
    proba_b = model_b.predict_proba(X_test_b_reshaped)[:, 1]
    proba_c = model_c.predict_proba(X_test_c_reshaped)[:, 1]
    
    X_meta_test = np.column_stack([proba_a, proba_b])
    y_pred_meta = model_meta.predict(X_meta_test)

    print(f'model predicted: {y_pred_meta[0]}')

    return y_pred_meta[0]


def meta_optimization(n_iter, param_func, df_train, df_test, model_a, model_b, model_c, model_meta, features_a, features_b, features_c, min_prec, min_rec):

    meta_models = []

    y_train = df_train['Trades']  
    y_test = df_test['Trades']   

    
    X_train_a = df_train[features_a]
    X_test_a = df_test[features_a]
    
    X_train_b = df_train[features_b]
    X_test_b = df_test[features_b]

    X_train_c = df_train[features_c]
    X_test_c = df_test[features_c]
    
    proba_train_a = model_a.predict_proba(X_train_a)[:, 1]
    proba_train_b = model_b.predict_proba(X_train_b)[:, 1]
    proba_train_c = model_c.predict_proba(X_train_c)[:, 1]

    proba_a = model_a.predict_proba(X_test_a)[:, 1]
    proba_b = model_b.predict_proba(X_test_b)[:, 1]
    proba_c = model_c.predict_proba(X_test_c)[:, 1]
    
    X_meta_train =  np.column_stack([proba_train_a, proba_train_b, proba_train_c])
    X_meta_test = np.column_stack([proba_a, proba_b, proba_c])


    for i in range(n_iter):

        
        params = param_func()

        model = model_meta(**params)

        model.fit(X_meta_train, y_train)

        y_pred = model.predict(X_meta_test)
        
        performance_df = meta_section_validation(validation_df = df_test,
                                            model_a = model_a,
                                            model_b = model_b,
                                            model_c = model_c,
                                            model_meta = model,
                                            features_a = features_a,
                                            features_b = features_b,
                                            features_c = features_c)
        precision_1 = performance_df['precision'].mean()
        recall_1 = performance_df['recall'].mean()
        precision_sec = performance_df['precision'].min()
        
        if precision_1 >= min_prec and recall_1 >= min_rec and precision_sec > 0.25:
            params = model.get_params()
            meta_models.append({
                    'model' : model_meta,
                    'params' : params,
                    'precision' : precision_1,
                    'recall' : recall_1,
                    'min_precision': precision_sec
                })
            print(f'model {i} meets criteria: precision = {precision_1}, recall = {recall_1}')

    return meta_models

In [1]:
class EnsembleModel():

    def __init__(self, models, ensembler, features_list):

        
        self.models = models
        self.features_list = features_list
        self.ensembler = ensembler


    def transform(self, X):

        probability_list = []

        for model, feature in zip(self.models, self.features_list):
            y_proba = model.predict_proba(X[feature])[:, 1]
            probability_list.append(y_proba)

        return np.column_stack(probability_list)
        
    def predict_proba(self, X):
        y_proba = self.transform(X)
        return self.ensembler.predict_proba(y_proba)
        
    def fit(self, X, y):
        y_proba = self.transform(X)
        self.ensembler.fit(y_proba, y)

        return self

    def predict(self, X):
        y_proba = self.transform(X)
        predictions = self.ensembler.predict(y_proba)
        return predictions

    def section_validation(self, step, df):
        
        performances = []
        tot_samples = len(df)
        
        for start in range(0, tot_samples, step):
            end = min(start + step, tot_samples)  
            
            section_df = df.iloc[start:end]
            y_section = section_df['Trades']
        
            y_pred = self.predict(section_df)
            y_proba = self.predict_proba(section_df)[:, 1]
     
            # Calculate metrics for this section only
            precision_section = precision_score(y_section, y_pred, pos_label=1)
            recall_section = recall_score(y_section, y_pred, pos_label=1)
            f1_values = f1_score(y_section, y_pred, average = 'macro')
            
            performances.append({
                'section': f"{start}-{end}",
                'precision': precision_section,
                'recall': recall_section,
                'f1_values': f1_values,
                'trades_in_section': y_section.sum(),
                'signals_generated': y_pred.sum(),
                'section_size': len(section_df)
            })
            
        return pd.DataFrame(performances)
 
                